# MSP-Podcast 破損候補の感情ラベル照合

`msp_unavailable_filenames.txt` と `labels_consensus.csv` をファイル名で照合し、元の感情ラベルと研究用4クラスへの対応を確認します。このNotebookは音声ファイルを検索・読み込み・再生せず、結果ファイルも書き出しません。

既定では合成データだけを使用します。実metadataを確認するときだけ、設定セルのパスを確認して `RUN_REAL_DATA = True` に変更してください。


In [13]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.contracts import LABEL_ORDER, map_emotion

# Safety gate: Falseのままなら、下の実ファイルは一切読みません。
RUN_REAL_DATA = False
RUN_WAV_CSV_AUDIT = True

RUN_WAV_CSV_AUDIT = True

MSP_LABEL_CSV_PATH = Path(
    r"C:\Users\RD004\Documents\lab\data\MSP_PODCAST\Labels\labels_consensus.csv"
)

MSP_AUDIO_DIR = Path(
    r"C:\Users\RD004\Documents\lab\data\MSP_PODCAST\Audio"
)


## 1. 読み込み・照合関数

照合キーは、パス部分を除いたファイル名の前後空白を除去し、大文字小文字を区別しない形に正規化します。表示結果には入力時の表記を残します。


In [14]:
def normalize_filename(value):
    """Return a case-insensitive basename key without touching the file itself."""
    text = str(value).strip().replace('\\', '/')
    return text.rsplit('/', 1)[-1].casefold()


def prepare_unavailable_lines(lines):
    """Normalize text-list rows and report blanks and duplicate candidate names."""
    raw_lines = [str(line) for line in lines]
    nonempty = [line.strip() for line in raw_lines if line.strip()]
    frame = pd.DataFrame({'input_filename': nonempty})
    frame['match_key'] = frame['input_filename'].map(normalize_filename)
    duplicate_mask = frame.duplicated('match_key', keep=False)
    duplicate_candidates = frame.loc[duplicate_mask, ['input_filename', 'match_key']].copy()
    unique_frame = frame.drop_duplicates('match_key', keep='first').reset_index(drop=True)
    stats = {
        'raw_line_count': len(raw_lines),
        'blank_line_count': len(raw_lines) - len(nonempty),
        'nonempty_input_rows': len(nonempty),
        'unique_input_files': len(unique_frame),
        'duplicate_input_rows': len(nonempty) - len(unique_frame),
        'non_wav_input_rows': int((~frame['match_key'].str.endswith('.wav')).sum()),
    }
    return unique_frame, stats, duplicate_candidates.reset_index(drop=True)


def read_unavailable_list(path):
    """Read only the named text file; no audio directory is inspected."""
    lines = Path(path).read_text(encoding='utf-8-sig').splitlines()
    return prepare_unavailable_lines(lines)


def prepare_label_frame(frame):
    """Validate and normalize MSP label metadata without resolving audio paths."""
    required = {'FileName', 'EmoClass'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'MSP label CSV is missing columns: {sorted(missing)}')

    labels = frame.loc[:, ['FileName', 'EmoClass']].copy()
    labels['FileName'] = labels['FileName'].astype(str).str.strip()
    labels['EmoClass'] = labels['EmoClass'].astype(str).str.strip()
    labels['match_key'] = labels['FileName'].map(normalize_filename)
    if labels['match_key'].eq('').any():
        raise ValueError('MSP label CSV contains an empty FileName')

    duplicate_mask = labels.duplicated('match_key', keep=False)
    if duplicate_mask.any():
        examples = labels.loc[duplicate_mask, 'FileName'].head(10).tolist()
        raise ValueError(f'MSP label CSV has duplicate normalized filenames: {examples}')
    return labels.rename(columns={'FileName': 'csv_filename', 'EmoClass': 'original_emotion'})


def read_msp_label_csv(path):
    """Read the MSP UTF-8/BOM-compatible metadata CSV as strings."""
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    return prepare_label_frame(frame)


In [15]:
def scan_wav_directory(audio_dir):
    """List WAV filenames recursively without opening or decoding audio."""
    root = Path(audio_dir)
    if not root.is_dir():
        raise FileNotFoundError(f'MSP audio directory was not found: {root}')

    rows = []
    for path in root.rglob('*'):
        if path.is_file() and path.suffix.casefold() == '.wav':
            rows.append(
                {
                    'wav_filename': path.name,
                    'wav_relative_path': path.relative_to(root).as_posix(),
                    'match_key': normalize_filename(path.name),
                }
            )
    wavs = pd.DataFrame(rows, columns=['wav_filename', 'wav_relative_path', 'match_key'])
    duplicate_mask = wavs.duplicated('match_key', keep=False)
    if duplicate_mask.any():
        examples = wavs.loc[duplicate_mask, 'wav_relative_path'].head(10).tolist()
        raise ValueError(f'MSP audio directory has duplicate normalized WAV filenames: {examples}')
    return wavs.sort_values('wav_relative_path').reset_index(drop=True)


def audit_wav_csv(wavs, labels):
    """Compare actual WAV filenames and label rows in both directions."""
    comparison = labels.merge(
        wavs,
        how='outer',
        on='match_key',
        validate='one_to_one',
        indicator=True,
    )
    comparison['comparison_status'] = comparison['_merge'].map(
        {
            'both': 'matched',
            'left_only': 'label_without_wav',
            'right_only': 'wav_without_label',
        }
    ).astype(str)
    comparison['original_emotion'] = comparison['original_emotion'].fillna('')
    label_present = comparison['_merge'].ne('right_only')
    mapping = pd.DataFrame(
        [mapping_fields(row.original_emotion, present) for row, present in zip(
            comparison.itertuples(index=False), label_present
        )]
    )
    comparison = pd.concat([comparison.reset_index(drop=True), mapping], axis=1)
    comparison = comparison.drop(columns=['_merge', 'match_key'])

    available = comparison.loc[comparison['comparison_status'].eq('matched')].copy()
    missing_wavs = comparison.loc[
        comparison['comparison_status'].eq('label_without_wav'),
        ['csv_filename', 'original_emotion', 'mapped_emotion', 'mapping_status'],
    ].reset_index(drop=True)
    unlabeled_wavs = comparison.loc[
        comparison['comparison_status'].eq('wav_without_label'),
        ['wav_filename', 'wav_relative_path', 'comparison_status'],
    ].reset_index(drop=True)

    summary = pd.DataFrame(
        [
            {
                'wav_files_found': len(wavs),
                'label_csv_rows': len(labels),
                'matched_wav_and_label': len(available),
                'label_rows_without_wav': len(missing_wavs),
                'wav_files_without_label': len(unlabeled_wavs),
                'available_primary_4_files': int(available['included_in_primary_4'].eq(True).sum()),
            }
        ]
    )
    raw_counts = (
        available['original_emotion']
        .replace('', '<empty_label>')
        .value_counts(dropna=False)
        .rename_axis('original_emotion')
        .reset_index(name='count')
    )
    mapped_counts = (
        available.loc[available['included_in_primary_4'].eq(True), 'mapped_emotion']
        .value_counts()
        .reindex(LABEL_ORDER, fill_value=0)
        .rename_axis('mapped_emotion')
        .reset_index(name='count')
    )
    return summary, raw_counts, mapped_counts, comparison, missing_wavs, unlabeled_wavs


In [16]:
def mapping_fields(original_emotion, matched):
    if not matched:
        return {
            'mapped_emotion': None,
            'included_in_primary_4': None,
            'mapping_status': 'metadata_not_found',
            'mapping_version': None,
        }
    if not original_emotion:
        return {
            'mapped_emotion': None,
            'included_in_primary_4': False,
            'mapping_status': 'empty_label',
            'mapping_version': None,
        }
    try:
        decision = map_emotion('msp_podcast', original_emotion)
    except ValueError:
        return {
            'mapped_emotion': None,
            'included_in_primary_4': False,
            'mapping_status': 'unknown_label',
            'mapping_version': None,
        }
    return {
        'mapped_emotion': decision.mapped_emotion,
        'included_in_primary_4': decision.included,
        'mapping_status': 'included_primary_4' if decision.included else 'not_in_primary_4',
        'mapping_version': decision.mapping_version,
    }


def audit_unavailable_labels(candidates, input_stats, labels):
    details = candidates.merge(labels, how='left', on='match_key', validate='one_to_one', indicator=True)
    details['metadata_match'] = details['_merge'].eq('both')
    details['original_emotion'] = details['original_emotion'].fillna('')
    mapping = pd.DataFrame(
        [mapping_fields(row.original_emotion, row.metadata_match) for row in details.itertuples(index=False)]
    )
    details = pd.concat([details.reset_index(drop=True), mapping], axis=1)
    details = details.drop(columns=['_merge', 'match_key'])

    summary_values = dict(input_stats)
    summary_values.update(
        {
            'label_csv_rows': len(labels),
            'metadata_matched_files': int(details['metadata_match'].sum()),
            'metadata_unmatched_files': int((~details['metadata_match']).sum()),
            'primary_4_files': int(details['included_in_primary_4'].eq(True).sum()),
            'outside_primary_4_files': int(details['mapping_status'].eq('not_in_primary_4').sum()),
            'empty_label_files': int(details['mapping_status'].eq('empty_label').sum()),
            'unknown_label_files': int(details['mapping_status'].eq('unknown_label').sum()),
        }
    )
    summary = pd.DataFrame([summary_values])

    raw_outcomes = details['original_emotion'].where(details['metadata_match'], '<metadata_not_found>')
    raw_outcomes = raw_outcomes.replace('', '<empty_label>')
    raw_label_counts = raw_outcomes.value_counts(dropna=False).rename_axis('original_emotion').reset_index(name='count')

    def mapped_outcome(row):
        if row.mapping_status == 'included_primary_4':
            return row.mapped_emotion
        return f'<{row.mapping_status}>'

    outcome_order = list(LABEL_ORDER) + [
        '<not_in_primary_4>', '<empty_label>', '<unknown_label>', '<metadata_not_found>'
    ]
    mapped_outcomes = details.apply(mapped_outcome, axis=1)
    mapped_label_counts = (
        mapped_outcomes.value_counts()
        .reindex(outcome_order, fill_value=0)
        .rename_axis('mapped_outcome')
        .reset_index(name='count')
    )
    unmatched = details.loc[
        ~details['metadata_match'], ['input_filename', 'mapping_status']
    ].reset_index(drop=True)
    return summary, raw_label_counts, mapped_label_counts, details, unmatched


## 2. 入力選択

`RUN_REAL_DATA = False` では、照合境界を確認する小さな合成例を使います。実metadataモードでも、音声ファイルにはアクセスしません。


In [17]:
if RUN_REAL_DATA:
    if MSP_LABEL_CSV_PATH is None:
        raise ValueError(
            'Set MSP_LABEL_CSV_PATH directly, or set MSP_LABEL_CSV_PATH/MSP_PODCAST_ROOT in the environment.'
        )
    candidates, input_stats, duplicate_candidates = read_unavailable_list(UNAVAILABLE_LIST_PATH)
    labels = read_msp_label_csv(MSP_LABEL_CSV_PATH)
    data_mode = 'real_metadata'
else:
    synthetic_lines = [
        ' MSP-PODCAST_DEMO_A.wav ',
        'MSP-PODCAST_DEMO_H.wav',
        'MSP-PODCAST_DEMO_S.wav',
        'MSP-PODCAST_DEMO_D.wav',
        'MSP-PODCAST_DEMO_EXCLUDED.wav',
        'MSP-PODCAST_DEMO_UNKNOWN.wav',
        'MSP-PODCAST_DEMO_EMPTY.wav',
        'MSP-PODCAST_DEMO_DUP.wav',
        'msp-podcast_demo_dup.WAV',
        'MSP-PODCAST_DEMO_MISSING.wav',
        '',
    ]
    synthetic_labels = pd.DataFrame(
        {
            'FileName': [
                'MSP-PODCAST_DEMO_A.wav', 'MSP-PODCAST_DEMO_H.wav',
                'MSP-PODCAST_DEMO_S.wav', 'MSP-PODCAST_DEMO_D.wav',
                'MSP-PODCAST_DEMO_EXCLUDED.wav', 'MSP-PODCAST_DEMO_UNKNOWN.wav',
                'MSP-PODCAST_DEMO_EMPTY.wav', 'MSP-PODCAST_DEMO_DUP.wav',
            ],
            'EmoClass': ['A', 'H', 'S', 'D', 'C', 'Z', '', 'H'],
        }
    )
    candidates, input_stats, duplicate_candidates = prepare_unavailable_lines(synthetic_lines)
    labels = prepare_label_frame(synthetic_labels)
    data_mode = 'synthetic_only'

summary, raw_label_counts, mapped_label_counts, details, unmatched = audit_unavailable_labels(
    candidates, input_stats, labels
)
{'data_mode': data_mode, 'run_real_data': RUN_REAL_DATA}


{'data_mode': 'synthetic_only', 'run_real_data': False}

## 3. 入力・照合サマリー


In [18]:
display(summary)
if not duplicate_candidates.empty:
    print('重複として1件にまとめた候補:')
    display(duplicate_candidates)


,raw_line_count,blank_line_count,nonempty_input_rows,unique_input_files,duplicate_input_rows,non_wav_input_rows,label_csv_rows,metadata_matched_files,metadata_unmatched_files,primary_4_files,outside_primary_4_files,empty_label_files,unknown_label_files
0,11,1,10,9,1,0,8,8,1,5,1,1,1


重複として1件にまとめた候補:


,input_filename,match_key
0,MSP-PODCAST_DEMO_DUP.wav,msp-podcast_demo_dup.wav
1,msp-podcast_demo_dup.WAV,msp-podcast_demo_dup.wav


## 4. 元ラベルと4クラス対応の件数


In [19]:
print('CSV元ラベル別件数:')
display(raw_label_counts)
print('4クラス対応・対象外・未一致別件数:')
display(mapped_label_counts)


CSV元ラベル別件数:


,original_emotion,count
0,H,2
1,A,1
2,S,1
3,D,1
4,C,1
5,Z,1
6,<empty_label>,1
7,<metadata_not_found>,1


4クラス対応・対象外・未一致別件数:


,mapped_outcome,count
0,anger,1
1,happy,2
2,sadness,1
3,disgust,1
4,<not_in_primary_4>,1
5,<empty_label>,1
6,<unknown_label>,1
7,<metadata_not_found>,1


## 5. ファイル単位の照合結果


In [20]:
detail_columns = [
    'input_filename', 'csv_filename', 'metadata_match', 'original_emotion',
    'mapped_emotion', 'included_in_primary_4', 'mapping_status', 'mapping_version',
]
display(details.loc[:, detail_columns])


,input_filename,csv_filename,metadata_match,original_emotion,mapped_emotion,included_in_primary_4,mapping_status,mapping_version
0,MSP-PODCAST_DEMO_A.wav,MSP-PODCAST_DEMO_A.wav,True,A,anger,True,included_primary_4,msp_podcast_r1_10_primary_v1
1,MSP-PODCAST_DEMO_H.wav,MSP-PODCAST_DEMO_H.wav,True,H,happy,True,included_primary_4,msp_podcast_r1_10_primary_v1
2,MSP-PODCAST_DEMO_S.wav,MSP-PODCAST_DEMO_S.wav,True,S,sadness,True,included_primary_4,msp_podcast_r1_10_primary_v1
3,MSP-PODCAST_DEMO_D.wav,MSP-PODCAST_DEMO_D.wav,True,D,disgust,True,included_primary_4,msp_podcast_r1_10_primary_v1
4,MSP-PODCAST_DEMO_EXCLUDED.wav,MSP-PODCAST_DEMO_EXCLUDED.wav,True,C,None,False,not_in_primary_4,msp_podcast_r1_10_primary_v1
5,MSP-PODCAST_DEMO_UNKNOWN.wav,MSP-PODCAST_DEMO_UNKNOWN.wav,True,Z,None,False,unknown_label,None
6,MSP-PODCAST_DEMO_EMPTY.wav,MSP-PODCAST_DEMO_EMPTY.wav,True,,None,False,empty_label,None
7,MSP-PODCAST_DEMO_DUP.wav,MSP-PODCAST_DEMO_DUP.wav,True,H,happy,True,included_primary_4,msp_podcast_r1_10_primary_v1
8,MSP-PODCAST_DEMO_MISSING.wav,NaN,False,,None,None,metadata_not_found,None


## 6. CSVに存在しなかった候補


In [21]:
if unmatched.empty:
    print('すべての候補がラベルCSVに存在しました。')
else:
    display(unmatched)


,input_filename,mapping_status
0,MSP-PODCAST_DEMO_MISSING.wav,metadata_not_found


## 7. 実際に存在するWAVとラベルCSVの照合

設定セルで `MSP_AUDIO_DIR` と `MSP_LABEL_CSV_PATH` を指定し、`RUN_WAV_CSV_AUDIT = True` にした場合だけ実行します。WAVはファイル名を列挙するだけで、音声内容を開いたりデコードしたりしません。


In [22]:
if RUN_WAV_CSV_AUDIT:
    if MSP_AUDIO_DIR is None:
        raise ValueError('Set MSP_AUDIO_DIR directly, or set MSP_AUDIO_DIR/MSP_PODCAST_ROOT in the environment.')
    if MSP_LABEL_CSV_PATH is None:
        raise ValueError(
            'Set MSP_LABEL_CSV_PATH directly, or set MSP_LABEL_CSV_PATH/MSP_PODCAST_ROOT in the environment.'
        )

    wav_files = scan_wav_directory(MSP_AUDIO_DIR)
    wav_labels = read_msp_label_csv(MSP_LABEL_CSV_PATH)
    (
        wav_csv_summary,
        available_raw_counts,
        available_mapped_counts,
        wav_csv_comparison,
        label_rows_without_wav,
        wav_files_without_label,
    ) = audit_wav_csv(wav_files, wav_labels)

    print('WAVとラベルCSVの照合サマリー:')
    display(wav_csv_summary)
    print('実在WAVの元ラベル別件数:')
    display(available_raw_counts)
    print('実在WAVの研究用4クラス別件数:')
    display(available_mapped_counts)
    print('CSVにはあるがWAVが見つからない行:')
    display(label_rows_without_wav)
    print('WAVはあるがCSVにラベルがないファイル:')
    display(wav_files_without_label)
else:
    print('WAVフォルダ照合は無効です。設定セルのRUN_WAV_CSV_AUDITをTrueにすると実行します。')


WAVとラベルCSVの照合サマリー:


,wav_files_found,label_csv_rows,matched_wav_and_label,label_rows_without_wav,wav_files_without_label,available_primary_4_files
0,25111,104267,25111,79156,0,25111


実在WAVの元ラベル別件数:


,original_emotion,count
0,H,15429
1,A,4579
2,S,2693
3,D,2410


実在WAVの研究用4クラス別件数:


,mapped_emotion,count
0,anger,4579
1,happy,15429
2,sadness,2693
3,disgust,2410


CSVにはあるがWAVが見つからない行:


,csv_filename,original_emotion,mapped_emotion,mapping_status
0,MSP-PODCAST_0001_0008.wav,N,None,not_in_primary_4
1,MSP-PODCAST_0001_0009.wav,N,None,not_in_primary_4
2,MSP-PODCAST_0001_0011.wav,N,None,not_in_primary_4
3,MSP-PODCAST_0001_0013.wav,N,None,not_in_primary_4
4,MSP-PODCAST_0001_0016.wav,N,None,not_in_primary_4
...,...,...,...,...
79151,MSP-PODCAST_3294_0045_0020.wav,U,None,not_in_primary_4
79152,MSP-PODCAST_3294_0045_0104.wav,H,happy,included_primary_4
79153,MSP-PODCAST_3294_0045_0120.wav,N,None,not_in_primary_4
79154,MSP-PODCAST_3294_0046_0075.wav,H,happy,included_primary_4


WAVはあるがCSVにラベルがないファイル:


,wav_filename,wav_relative_path,comparison_status
